# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-08','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item from one order

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order','vendor_id'])
# TODO: assert the row count and the quantity total
assert len(lines) == 4
assert lines['qty'].sum() == 7
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-08


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# turn vendors into dataframe
vendors = pd.DataFrame(vendors_json)
display(vendors)

# join vendor information
joined = lines.merge(
    vendors,
    on = 'vendor_id',
    how = 'left',
    indicator = True
)
display(joined)

# determine which vendor did not match and how many units attatched to it
unmatched = joined[joined['_merge']=='left_only']
print('unmatched vendor(s):', unmatched['vendor_id'].unique())
print('units attached to unmatched vendor(s):', unmatched['qty'].sum())

,vendor_id,name,zone
0,V-01,Hoos Burgers,A
1,V-10,Cav Merch,A


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-08,NaN,NaN,left_only


unmatched vendor(s): ['V-08']
units attached to unmatched vendor(s): 3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# keep unmatched vendors in the table and label as unknown
# this way we keep the total sales value accurate

joined['name'] = joined['name'].fillna('Unknown vendor')

vendor_totals = joined.groupby('name', as_index = False).agg(total_qty = ('qty','sum'))
display(vendor_totals)

,name,total_qty
0,Cav Merch,1
1,Hoos Burgers,3
2,Unknown vendor,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# find the information
zones = joined.groupby('zone')
zone_totals = zones['qty'].sum()
zone_sum = zone_totals.sum()
zone_max = zone_totals.idxmax()

# print results
display(zone_totals)
print("Zone sum: ", zone_sum)
print("Zone max: ", zone_max)
print("Because the unmatched vendor V-08 has no zone, 3 units are exclude from groupby('zone').")

,qty
zone,
A,4


Zone sum:  4
Zone max:  A
Because the unmatched vendor V-08 has no zone, 3 units are exclude from groupby('zone').


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

safe_ragged = []

for order in ragged_json:
    safe_order = order.copy()
    safe_order['lines'] = order.get('lines', [])
    safe_ragged.append(safe_order)

ragged_lines = pd.json_normalize(
    safe_ragged,
    record_path = 'lines',
    meta = ['order','vendor_id']
)
display(ragged_lines)

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** Having one row per item allows a user to analyze indivudual items easier since each item would have its own vendor and quantity information. For example if you wanted to find the total quanity sold of an item it is much easier this way.

**b)** A missing value indicates that the value was not reported, whereas a zero is a real value indicating that zero units were sold. If you start treating missing values as zeros or zeros as missing values, you make averages and other statistical summaries misleading.

